# Sais: íons, neutralização e concentração
### Química — funções inorgânicas

**Objetivo:** Identificar sais, escrever sua formação e calcular a massa para uma solução de NaCl.

### Como usar este notebook
Execute as células em ordem. As de código conferem as contas e geram as visualizações. No 3D, **arraste com o mouse para girar em 360°** e use a roda para zoom.
A primeira célula instala somente dependências ausentes. O 3D precisa de acesso a `cdn.jsdelivr.net` no navegador e de um notebook confiável (trusted); se a rede bloquear o visualizador, o desenho 2D continua disponível.

Material para estudo teórico: não prove, toque nem misture produtos químicos para identificar uma função química.

In [ ]:
# Preparação idempotente: usa o Python do próprio kernel.
import importlib.util
import subprocess
import sys

for pacote, modulo in [("rdkit", "rdkit"), ("py3Dmol", "py3Dmol"),
                       ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pacote], check=True)

import math
import matplotlib.pyplot as plt
import py3Dmol
from rdkit import Chem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG, display

## 1. Conceito principal

Sais são compostos iônicos formados por cátions e ânions. Na abordagem escolar de Arrhenius, sua dissociação fornece pelo menos um cátion diferente de H⁺ e um ânion diferente de OH⁻. Podem ser obtidos por neutralização:

$$\mathrm{HCl(aq)+NaOH(aq)\rightarrow NaCl(aq)+H_2O(l)}$$
$$\boxed{n=cV,\qquad m=nM=cVM}$$

| Fórmula | Nome | Balanço de cargas |
|---|---|---|
| NaCl | cloreto de sódio | (+1) + (−1) = 0 |
| CaCl₂ | cloreto de cálcio | (+2) + 2(−1) = 0 |
| Na₂SO₄ | sulfato de sódio | 2(+1) + (−2) = 0 |
| NaHCO₃ | hidrogenocarbonato de sódio | (+1) + (−1) = 0 |

O nome vem do ânion seguido de “de” + cátion. Sais **não são sempre solúveis nem sempre produzem pH 7**: Na₂CO₃ produz solução básica; NH₄Cl, ácida. NaCl é aproximadamente neutro em água pura a 25 °C.

## 2. Exemplo resolvido passo a passo

Cálculo teórico para $250\ \mathrm{mL}$ de solução de NaCl a $0{,}100\ \mathrm{mol/L}$, usando $M=58{,}44\ \mathrm{g/mol}$.

$$V=250/1000=0{,}250\ \mathrm{L}$$
$$n=0{,}100\cdot0{,}250=0{,}0250\ \mathrm{mol}$$
$$m=0{,}0250\cdot58{,}44=\boxed{1{,}461\ \mathrm{g}}$$

O volume é o da **solução final**, não o volume inicial de água.

## 3. Conferindo o cálculo com Python

In [ ]:
c, volume, massa_molar = 0.100, 0.250, 58.44
n = c * volume
massa = n * massa_molar
print(f"n = {n:.4f} mol; massa de NaCl = {massa:.3f} g")
assert math.isclose(massa, 1.461)

## 4. Visualização

### 4.1 Modelo 3D interativo

**Fragmento didático da rede iônica do NaCl.** Na⁺ = violeta e Cl⁻ = verde, em posições alternadas com separação de 2,82 Å entre vizinhos de cargas opostas. Não há moléculas discretas de NaCl nem varetas covalentes Na–Cl. As bordas do fragmento cortam a rede; no cristal ideal, cada íon tem seis vizinhos imediatos de carga oposta. Os tamanhos das esferas são apenas ilustrativos. No 2D, os íons ficam separados e com cargas explícitas.

In [ ]:
# Rede recortada, sem ligações covalentes artificiais.
from itertools import product
visor = py3Dmol.view(width=640, height=400)
for i, j, k in product(range(4), repeat=3):
    sodio = (i + j + k) % 2 == 0
    visor.addSphere({"center": {"x": i * 2.82, "y": j * 2.82, "z": k * 2.82},
                     "radius": 0.65 if sodio else 1.0,
                     "color": "#AB5CF2" if sodio else "#1FF01F"})
visor.setBackgroundColor("#ffffff")
visor.zoomTo()
visor.show()

mol = Chem.MolFromSmiles("[Na+].[Cl-]")
if mol is None:
    raise ValueError("Representação dos íons inválida.")

### 4.2 Representação 2D

Íons separados: não há ligação covalente a desenhar neste sal.

In [ ]:
# Estrutura 2D: ligações covalentes com traços; íons separados por ponto
# no SMILES permanecem separados, sem inventar uma ligação covalente.
plano = Chem.Mol(mol)
rdDepictor.Compute2DCoords(plano)
desenho = rdMolDraw2D.MolDraw2DSVG(520, 300)
desenho.drawOptions().bondLineWidth = 2.2
desenho.DrawMolecule(plano)
desenho.FinishDrawing()
display(SVG(desenho.GetDrawingText()))

### 4.3 Gráfico de dados

In [ ]:
volumes = [0.050 * k for k in range(1, 11)]
plt.plot([v * 1000 for v in volumes], [0.100 * v * 58.44 for v in volumes], "o-")
plt.xlabel("Volume final de solução (mL)")
plt.ylabel("Massa de NaCl (g)")
plt.title("Solução a 0,100 mol/L: massa proporcional ao volume")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Como interpretar o resultado

A rede mostra alternância de cargas, não pares moleculares isolados. Em água, a dissociação separa os íons hidratados. No gráfico, dobrar o volume final mantendo a concentração dobra a massa de sal.

## 6. Exercício de fixação

Qual massa de NaCl corresponde a 500 mL de solução a 0,200 mol/L? Quantos mol de Na⁺ e de Cl⁻ há, admitindo dissociação completa?

Tente resolver antes de executar a resposta abaixo.

In [ ]:
n_ex = 0.200 * 0.500
m_ex = n_ex * 58.44
print(f"Massa = {m_ex:.3f} g; Na+ = {n_ex:.3f} mol; Cl- = {n_ex:.3f} mol")
assert math.isclose(m_ex, 5.844)

## 7. Resumo

- Fórmulas de sais respeitam a neutralidade elétrica.
- NaCl é uma rede iônica, não um conjunto de moléculas Na–Cl.
- m = cVM, com volume em litros.
- Solubilidade e pH dependem dos íons do sal.

## Referências

- IUPAC. *Compendium of Chemical Terminology (Gold Book)*: https://goldbook.iupac.org/
- ATKINS, P.; JONES, L. *Princípios de Química*. Bookman.